In [2]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO

In [3]:
def diff_calculation(SChar, VChar, type, mutate_matrix):
    matrix_values = mutate_matrix.values
    aa_to_idx = {aa: i for i, aa in enumerate(mutate_matrix.columns)}
    GAP = '-'

    difference = []
    for a, b in zip(SChar, VChar):
        if a == GAP or b == GAP or a not in aa_to_idx or b not in aa_to_idx:
            difference.append(0)
            continue

        if type == 'one-hot':
            diff = 0 if a == b else 1
        elif type == 'mut-mat':
            ia, ib = aa_to_idx[a], aa_to_idx[b]
            s_xx = matrix_values[ia, ia]
            s_yy = matrix_values[ib, ib]
            s_xy = matrix_values[ia, ib]
            diff = s_xx + s_yy - 2 * s_xy
        else:
            raise ValueError(f"未知类型: {type}")

        difference.append(diff)

    return difference

def pair_representation(serumHA, virusHA, type, mutate_matrix):
    serumChar = [char for char in serumHA]
    virusChar = [char for char in virusHA]

    if len(serumChar) != len(virusChar):
        return [np.nan for _ in range(len(serumChar))]

    return diff_calculation(serumChar, virusChar, type=type, mutate_matrix=mutate_matrix)

def split_data_by_strain(dataframe, identity_cols, frac):
    strains = dataframe[identity_cols].drop_duplicates().reset_index(drop=True)
    test_strains = strains.sample(frac=frac, random_state=42).reset_index(drop=True)
    test_strains_set = set(zip(*test_strains[identity_cols].values.T))

    mask = dataframe[identity_cols].apply(
        lambda row: tuple(row) in test_strains_set, axis=1)
    test_data = dataframe[mask]
    train_data = dataframe[~mask]

    return train_data, test_data

def rename_fasta_seq(input_file, sequence_data, output_file):
    records = list(SeqIO.parse(input_file, "fasta"))
    new_records = []
    for record, seq in zip(records, sequence_data):
        record.id = str(seq)
        record.description = ""
        new_records.append(record)
    SeqIO.write(new_records, output_file, "fasta")
    

In [4]:
mut_mat_path = './GIAG010101.csv'
mutate_matrix = pd.read_csv(mut_mat_path, index_col=0)
# meta feature for model training
meta_features = ['virusName',   # virus avidity (based on both name and passage)
                 # antiserum potency (based on both name and passage)
                 'serumName',
                 'virusPassCat',   # virus passage category
                 'serumPassCat']   # serum passage category


In [16]:
Crick_data = pd.read_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/data/data_40/Crick_41.csv', index_col=False)
# Crick_H1N1 = pd.read_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/data/new_try/data4model(Crick-H1N1).csv')
# Crick_H3N2 = pd.read_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/data/new_try/data4model(Crick-H3N2).csv')
# Crick_data = pd.concat([Crick_H1N1, Crick_H3N2], axis=0).drop_duplicates().reset_index(drop=True)

In [17]:
# H1_dt = Crick_data[Crick_data['Type'] == 'H1N1']
# H1_sequence = pd.concat([H1_dt['seq_a'], H1_dt['seq_c']], axis=0).reset_index(drop=True).drop_duplicates().reset_index(drop=True)
# H1_seq = [SeqRecord(Seq(seq), id=str(seq), description="") for seq in H1_sequence]
# output_file = "./H1_input.fasta"
# SeqIO.write(H1_seq, output_file, "fasta")
# # mafft --thread 12 --auto --inputorder "H1_input.fasta" > "H1_output.fasta"
# rename_fasta_seq('H1_41_output.fasta', H1_sequence,'H1_41_output.fasta')

# H3_dt = Crick_data[Crick_data['Type'] == 'H3N2']
# H3_sequence = pd.concat([H3_dt['seq_a'], H3_dt['seq_c']], axis=0).reset_index(drop=True).drop_duplicates().reset_index(drop=True)
# H3_seq = [SeqRecord(Seq(seq), id=str(seq), description="") for seq in H3_sequence]
# output_file = "./H3_input.fasta"
# SeqIO.write(H3_seq, output_file, "fasta")
# # mafft --thread 12 --auto --inputorder "H3_input.fasta" > "H3_output.fasta"
# rename_fasta_seq('H3_output.fasta', H3_sequence,'H3_output.fasta')


In [18]:
from Bio import SeqIO
import warnings

need_map = False
# 1. add mapped columns
# H1_dict = {rec.id: str(rec.seq)
#            for rec in SeqIO.parse("H1_output.fasta", "fasta")}
# H3_dict = {rec.id: str(rec.seq)
#            for rec in SeqIO.parse("H3_output.fasta", "fasta")}
# merged_dict = H1_dict | H3_dict
# Crick_data['serumHA'] = Crick_data['seq_a'].map(merged_dict)
# Crick_data['virusHA'] = Crick_data['seq_c'].map(merged_dict)
if(need_map):
    Crick_data['serumHA'] = Crick_data['seq_a'].map(merged_dict)
    Crick_data['virusHA'] = Crick_data['seq_c'].map(merged_dict)
else:
    Crick_data['serumHA'] = Crick_data['seq_a']
    Crick_data['virusHA'] = Crick_data['seq_c']

missing_a = Crick_data.loc[Crick_data['serumHA'].isna(), 'seq_a'].unique()
missing_c = Crick_data.loc[Crick_data['virusHA'].isna(), 'seq_c'].unique()
if len(missing_a) or len(missing_c):
    warnings.warn(
        "以下 seq_id 在 FASTA 中未找到，映射为 NaN:\n"
        f"  serumHA 缺失: {list(missing_a)}\n"
        f"  virusHA 缺失: {list(missing_c)}"
    )

# 2. remove space in Name
Crick_data['serumName'] = Crick_data['serumName'].str.replace(
    ' ', '', regex=False)
Crick_data['virusName'] = Crick_data['virusName'].str.replace(
    ' ', '', regex=False)

# 3. calculate sequence difference
diff_mat_list = []
diff_ohe_list = []
for ref_seq, test_seq in tqdm(zip(Crick_data['serumHA'], Crick_data['virusHA']), total=len(Crick_data), desc="Processing sequences"):
    if not ref_seq or not test_seq or pd.isna(ref_seq) or pd.isna(test_seq):
        diff_mat_list.append(np.nan)
        diff_ohe_list.append(np.nan)
        continue

    diff_mat_list.append(pair_representation(
        ref_seq, test_seq, type='mut-mat', mutate_matrix=mutate_matrix))
    diff_ohe_list.append(pair_representation(
        ref_seq, test_seq, type='one-hot', mutate_matrix=mutate_matrix))

Crick_data["seq_diff_mat"] = diff_mat_list
Crick_data["seq_diff_ohe"] = diff_ohe_list


Processing sequences:   0%|          | 0/2916 [00:00<?, ?it/s]

Processing sequences: 100%|██████████| 2916/2916 [00:01<00:00, 1855.24it/s]


In [20]:
dd = pd.read_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/data/data_40/Crick_41.csv')

In [27]:
dd.iloc[:, :4].equals(Crick_data.iloc[:, :4])

True

In [28]:
Crick_data.to_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/data/data_40/Crick_41_mapped.csv', index=False)


In [6]:
Crick_data = pd.read_csv('./all.csv')

FileNotFoundError: [Errno 2] No such file or directory: './all.csv'

In [45]:
serum_df = Crick_data[['seq_id_a', 'seq_id_b', 'seq_a', 'seq_b',
                       'serumPassCat', 'serumName', 'serumDate', 'Type', 'serumIslID']].copy()
virus_df = Crick_data[['seq_id_c', 'seq_id_d', 'seq_c', 'seq_d',
                       'virusPassCat', 'virusName', 'virusDate', 'Type', 'virusIslID']].copy()

columns = ['seq_id_a', 'seq_id_b', 'seq_a', 'seq_b',
           'serumPassCat', 'serumName', 'serumDate', 'Type', 'serumIslID']
serum_df.columns = columns
virus_df.columns = columns
strain_df = pd.concat([serum_df, virus_df], axis=0).drop_duplicates(subset=['seq_a', 'seq_b', 'serumPassCat'])

Artificial_data = pd.concat([strain_df, strain_df], axis=1)
Artificial_data.columns = ['seq_id_a', 'seq_id_b', 'seq_a', 'seq_b', 'serumPassCat', 'serumName', 'serumDate', 'Type', 'serumIslID',
                           'seq_id_c', 'seq_id_d', 'seq_c', 'seq_d', 'virusPassCat', 'virusName', 'virusDate', 'virusType', 'virusIslID']
Artificial_data = Artificial_data[['seq_id_a', 'seq_id_b', 'seq_id_c', 'seq_id_d', 'seq_a', 'seq_b', 'seq_c', 'seq_d',
                                   'serumPassCat', 'virusPassCat', 'serumName', 'virusName', 'serumDate', 'virusDate',
                                   'serumIslID', 'virusIslID', 'Type']]

Artificial_data.to_csv('./data_40/Artificial.csv', index=False)


In [46]:
group_columns = ['seq_a', 'seq_b', 'seq_c', 'seq_d', 'serumPassCat', 'virusPassCat']
agg_dict = {c: 'first' for c in Crick_data.columns if c not in group_columns}
agg_dict['label'] = 'mean'
Crick_data_final = Crick_data.groupby(group_columns).agg(agg_dict).reset_index()

### split by titer

In [47]:
# titer_train, titer_test = train_test_split(Crick_data, test_size=0.1, random_state=42)
titer_train, titer_test = train_test_split(Crick_data_final, test_size=0.1, random_state=42)
print(len(titer_train))
print(len(titer_test))
print(len(titer_train.loc[titer_train['Type'] == 'H1N1']))
print(len(titer_train.loc[titer_train['Type'] == 'H3N2']))
print(len(titer_test.loc[titer_test['Type'] == 'H1N1']))
print(len(titer_test.loc[titer_test['Type'] == 'H3N2']))


63120
7014
32860
30260
3682
3332


In [48]:
# titer_train.to_csv('./data/titer_split/train.csv')
# titer_test.to_csv('./data/titer_split/test.csv')
titer_train.to_csv('./data_40/titer/train.csv')
titer_test.to_csv('./data_40/titer/test.csv')

### split by strain

In [49]:
Identity_cols = ['seq_id_c', 'seq_id_d', 'virusPassCat', 'virusName']
strain_train, strain_test = split_data_by_strain(Crick_data_final, Identity_cols, 0.1)
print(len(strain_train))
print(len(strain_test))
print(len(strain_train.loc[strain_train['Type'] == 'H1N1']))
print(len(strain_train.loc[strain_train['Type'] == 'H3N2']))
print(len(strain_test.loc[strain_test['Type'] == 'H1N1']))
print(len(strain_test.loc[strain_test['Type'] == 'H3N2']))


63227
6907
33051
30176
3491
3416


In [50]:
# strain_train.to_csv('./data/strain_split/train.csv')
# strain_test.to_csv('./data/strain_split/test.csv')
strain_train.to_csv('./data_40/strain/train.csv')
strain_test.to_csv('./data_40/strain/test.csv')


### split by serum

In [1]:
Identity_cols = ['seq_id_a', 'seq_id_b', 'serumPassCat', 'serumName']
serum_train, serum_test = split_data_by_strain(Crick_data_final, Identity_cols, 0.065)
print(len(serum_train))
print(len(serum_test))
print(len(serum_train.loc[serum_train['Type'] == 'H1N1']))
print(len(serum_train.loc[serum_train['Type'] == 'H3N2']))
print(len(serum_test.loc[serum_test['Type'] == 'H1N1']))
print(len(serum_test.loc[serum_test['Type'] == 'H3N2']))

NameError: name 'split_data_by_strain' is not defined

In [52]:
# serum_train.to_csv('./data/serum_split/train.csv')
# serum_test.to_csv('./data/serum_split/test.csv')
serum_train.to_csv('./data_40/serum/train.csv')
serum_test.to_csv('./data_40/serum/test.csv')
